# 01. 전문가 중요도 측정 (A100 80GB 또는 G4, 약 1시간 / 약 8~10유닛)
ERNIE-4.5-21B-A3B(층마다 전문가 64개)에 **한국어 데이터**(위키 50% + 지시문 대화 50%, 약 26만 토큰)를 흘려 보내며,
층마다 전문가가
- 얼마나 자주 뽑히는지 (빈도, `freq`)
- 뽑혔을 때 출력에 얼마나 크게 기여하는지 (REAP 점수, `reap`)

를 기록합니다. 결과는 작은 JSON 파일(`results/calib/stats-ko.json`)이고 드라이브에 저장됩니다.
원본 BF16이 약 44GB라 A100 80GB 또는 G4가 필요합니다.

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 50, "런타임 > 런타임 유형 변경에서 A100 (80GB) 또는 G4 를 고르세요"

In [ ]:
!python scripts/01_calibrate.py --sets ko
push_results()

## 한국어는 소수의 전문가만 쓸까? (보고서용 그림)
층마다 REAP 점수가 높은 순으로 전문가를 늘어놓고, 상위 k개가 전체 점수의 몇 %를 차지하는지 봅니다.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
st = json.load(open("results/calib/stats-ko.json"))
shares = []
for layer in st["layers"].values():
    s = np.sort(np.array(layer["reap"]) * np.array(layer["freq"]))[::-1]   # 기여도 = 점수 x 빈도
    shares.append(np.cumsum(s) / s.sum())
shares = np.array(shares)
plt.plot(range(1, shares.shape[1] + 1), shares.mean(0), marker=".")
for k in (32, 48): plt.axvline(k, ls="--", c="gray"); print(f"상위 {k}개가 평균 {shares.mean(0)[k-1]*100:.1f}% 차지")
plt.xlabel("kept experts (sorted)"); plt.ylabel("cumulative share of Korean contribution"); plt.grid(alpha=.3)
plt.savefig("results/fig_korean_expert_share.png", dpi=150); plt.show()
push_results()